# Similarity Score Generation
This notebook takes the candidate pairs generated from union blocking and calculates similarity features for each pair.

- The input file is: `candidate_pairs_with_details.parquet`
- The output file is : `candidate_pairs_with_similarity.csv`

Similarity features created in this file include:
- First-name similarity
- Last-name similarity
- Full-name similarity
- Gender compatibility
- Birth-year difference
- Death-year difference
- Event-year difference
- Place match indicators
- Role-group comparison


In [1]:
!pip install rapidfuzz

In [2]:
# import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import fuzz


Define input and output paths

In [31]:
input_path = "../data/interim/candidate_pairs_with_details.parquet"
output_path = "../data/interim/Vamsi/candidate_pairs_with_similarity_scores.parquet"

print(f"Reading candidate pairs from {input_path}...")
print(f"Writing candidate pairs with similarity scores to {output_path}...")

Reading candidate pairs from ../data/interim/candidate_pairs_with_details.parquet...
Writing candidate pairs with similarity scores to ../data/interim/Vamsi/candidate_pairs_with_similarity_scores.parquet...


Load candidate pairs with details

In [4]:
candidate_pairs=pd.read_parquet(input_path)
print(f"The shape of the candidate pairs: {candidate_pairs.shape}")
candidate_pairs.head(5)

The shape of the candidate pairs: (1257724, 46)


,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count,event_idno_1,original_identifier_1,source_event_type_1,persona_type_1,role_group_1,name_1,...,gender_2,event_year_2,birth_year_2,death_year_2,event_place_clean_2,birth_place_clean_2,death_place_clean_2,resident_in_clean_2,matching_evidence_level_2,cleaning_issue_flags_2
0,persona-10,persona-10078,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1838.0,NaN,NaN,pampamarca santa iglesia,unknown,unknown,unknown,medium,no_major_issue
1,persona-10,persona-1008,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,female,1793.0,1793.0,NaN,unknown,unknown,unknown,unknown,medium,no_major_issue
2,persona-10,persona-10081,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1838.0,NaN,NaN,ishua santa iglesia,ishua,unknown,unknown,high,no_major_issue
3,persona-10,persona-10320,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1837.0,NaN,NaN,pampamarca santa iglesia,unknown,unknown,unknown,medium,no_major_issue
4,persona-10,persona-10366,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1896.0,NaN,NaN,pampamarca santa iglesia viceparroquial,unknown,unknown,unknown,medium,no_major_issue


Validate the required columns

In [5]:
required_columns = [
    "persona_idno_1",
    "persona_idno_2",

    "first_name_clean_1",
    "first_name_clean_2",

    "lastname_clean_1",
    "lastname_clean_2",

    "full_name_clean_1",
    "full_name_clean_2",

    "gender_1",
    "gender_2",

    "birth_year_1",
    "birth_year_2",

    "death_year_1",
    "death_year_2",

    "event_year_1",
    "event_year_2",

    "event_place_clean_1",
    "event_place_clean_2",

    "birth_place_clean_1",
    "birth_place_clean_2",

    "death_place_clean_1",
    "death_place_clean_2",

    "resident_in_clean_1",
    "resident_in_clean_2",

    "role_group_1",
    "role_group_2"
]

missing_columns = [
    col for col in required_columns
    if col not in candidate_pairs.columns
]

if len(missing_columns) == 0:
    print("All required columns are present.")
else:
    print("Missing columns:")
    print(missing_columns)

All required columns are present.


## Fuzzy Name Similarity:
We will check 3 similarity scores for names:
- Part A : First_name_similarity
- Part B : lastname_similarity
- Part C : fullname_similarity

These scores help identify pairs that may refer to the same person even when spelling is slightly different.

In [6]:
def fuzzy_score(value1,value2):
    """
    calculate the fuzzy similarity between 2 values

    returns:
    - score from 0 to 100 if bith values are present
    - 0 if either of the value is missing. 
    
    """
    if pd.isna(value1) or pd.isna(value2):
        return 0

    value1 = str(value1).strip().lower()
    value2 = str(value2).strip().lower()

    if value1=="unknown" or value2=="unknown":
        return 0
    if value1=="" or value2=="":
        return 0
    
    return fuzz.WRatio(value1,value2)

This function will compare 2 texts and provide the similarity scores from 0 to 100.

### Apply this function on candidate_pairs

In [7]:
candidate_pairs["first_name_similarity"]=[
    fuzzy_score(a,b) for a,b in zip(
        candidate_pairs["first_name_clean_1"],
        candidate_pairs["first_name_clean_2"]
    )
]
candidate_pairs["lastname_similarity"]=[
    fuzzy_score(a,b) for a,b in zip(
        candidate_pairs["lastname_clean_1"],
        candidate_pairs["lastname_clean_2"]
    )
]
candidate_pairs["full_name_similarity"]=[
    fuzzy_score(a,b) for a,b in zip(
        candidate_pairs["full_name_clean_1"],
        candidate_pairs["full_name_clean_2"]
    )
]

In [10]:
candidate_pairs[
    [
        "first_name_similarity",
        "lastname_similarity",
        "full_name_similarity"
    ]
].describe().round(2)

,first_name_similarity,lastname_similarity,full_name_similarity
count,1257724.00,1257724.00,1257724.00
mean,60.16,99.19,80.12
std,25.14,6.24,11.89
min,0.00,0.00,44.00
25%,40.00,100.00,70.97
50%,51.43,100.00,77.78
75%,83.33,100.00,88.89
max,100.00,100.00,100.00


Lastname similarity is very high because blocking is lastname-based.
First name similarity varies a lot.
Full name similarity is mostly medium to high.

## Gender Compatibility
- There are 5 types of genders `male`,`mostly_male`,`female`,`mostly_female`,`unknown`
- now we are categorizing the genders as 0 and 1 
-  `1 if compatible or unknown` , `0 if clearly incompatible`

In [11]:
def gender_compatibility(gender1,gender2):
    """ 
        Check whether two gender values are compatible.

    Returns:
    1 if compatible or unknown
    0 if clearly incompatible

    """

    if pd.isna(gender1) or pd.isna(gender2):
        return 1

    gender1 = str(gender1).strip().lower()
    gender2 = str(gender2).strip().lower()

    malevalues=["male","mostly_male"]
    femalevalues=["female","mostly_female"]
    unknownvalues=["unknown",""]

    if gender1 in unknownvalues or gender2 in unknownvalues:
        return 1
    if (gender1 in malevalues and gender2 in femalevalues) or (gender1 in femalevalues and gender2 in malevalues):
        return 0
    if (gender1 in malevalues and gender2 in malevalues) or (gender1 in femalevalues and gender2 in femalevalues):
        return 1
    

In [15]:
candidate_pairs["gender_similarity"] = [
    gender_compatibility(a,b) for a,b in zip(
        candidate_pairs["gender_1"],
        candidate_pairs["gender_2"]
    )
]

print("Total genders",candidate_pairs["gender_similarity"].value_counts().sum())
candidate_pairs["gender_similarity"].value_counts()

Total genders 1257724


gender_similarity
1    864005
0    393719
Name: count, dtype: int64

864,005 pairs → no gender conflict (About 68.7% pairs are gender-compatible/unknown)
393,719 pairs → gender conflict(31.3% pairs have gender conflict)

The gender similarity feature marks pairs as compatible when both genders match or when gender information is missing for at least one record. Clear gender conflicts are marked as 0

## Year Difference

This function calculates the difference between two years.

For example:
1790 and 1792 gives a difference of 2.

If either year is missing, the result is NaN.

In [16]:
def year_difference(year1,year2):
    """
    Calculate the absolute difference between two years.

    Returns:
    - absolute difference if both years are present
    - np.nan if either year is missing
    """
    if pd.isna(year1) or pd.isna(year2):
        return np.nan

    return abs(year1 - year2)

In [17]:
# apply them on candidate pairs
candidate_pairs["birth_year_difference"] = [
    year_difference(a,b) for a,b in zip(
        candidate_pairs["birth_year_1"],
        candidate_pairs["birth_year_2"]
    )
]
candidate_pairs["death_year_difference"] = [
    year_difference(a,b) for a,b in zip(
        candidate_pairs["death_year_1"],
        candidate_pairs["death_year_2"]
    )
]
candidate_pairs["event_year_difference"] = [
    year_difference(a,b) for a,b in zip(
        candidate_pairs["event_year_1"],
        candidate_pairs["event_year_2"]
    )
]

In [18]:
candidate_pairs[
    [
        "birth_year_difference",
        "death_year_difference",
        "event_year_difference"
    ]
].describe().round(2)

,birth_year_difference,death_year_difference,event_year_difference
count,49173.00,2650.00,1256783.00
mean,31.82,17.57,37.75
std,31.46,15.69,31.70
min,0.00,0.00,0.00
25%,4.00,3.00,11.00
50%,22.00,14.00,28.00
75%,54.00,28.75,60.00
max,169.00,71.00,160.00


## Place Match Features
Check whether two cleaned place values match exactly.

    Returns:
    - 1 if places match
    - 0 if places do not match
    NaN if either place is missing or unknown

In [19]:
def place_similarity(place1,place2):
    if pd.isna(place1) or pd.isna(place2):
        return np.nan

    place1 = str(place1).strip().lower()
    place2 = str(place2).strip().lower()

    if place1 == "" or place2 == "":
        return np.nan

    if place1 == "unknown" or place2 == "unknown":
        return np.nan

    return 1 if place1 == place2 else 0

In [21]:
candidate_pairs["event_place_similarity"] = [
    place_similarity(a,b) for a,b in zip(
        candidate_pairs["event_place_clean_1"],
        candidate_pairs["event_place_clean_2"]
    )
] 
candidate_pairs["birth_place_similarity"] = [
    place_similarity(a,b) for a,b in zip(
        candidate_pairs["birth_place_clean_1"],
        candidate_pairs["birth_place_clean_2"]
    )]
candidate_pairs["death_place_similarity"] = [
    place_similarity(a,b) for a,b in zip(   
        candidate_pairs["death_place_clean_1"],
        candidate_pairs["death_place_clean_2"]
    )]
candidate_pairs["resident_in_similarity"] = [
    place_similarity(a,b) for a,b in zip(
        candidate_pairs["resident_in_clean_1"],
        candidate_pairs["resident_in_clean_2"]
    )]

In [22]:
candidate_pairs[
    [
        "event_place_similarity",
        "birth_place_similarity",
        "death_place_similarity",
        "resident_in_similarity"
    ]
].apply(pd.Series.value_counts)

,event_place_similarity,birth_place_similarity,death_place_similarity,resident_in_similarity
0.0,618890,11019,736,510
1.0,419913,9050,767,356


## Role Group Comparison

In [23]:
candidate_pairs["same_role_group"] = np.where(
    candidate_pairs["role_group_1"] == candidate_pairs["role_group_2"],
    1,
    0
)

candidate_pairs["role_pair"] = (
    candidate_pairs["role_group_1"].astype(str)
    + "_to_"
    + candidate_pairs["role_group_2"].astype(str)
)

candidate_pairs[["role_group_1", "role_group_2", "same_role_group", "role_pair"]].head()

,role_group_1,role_group_2,same_role_group,role_pair
0,family_relation,family_relation,1,family_relation_to_family_relation
1,family_relation,main_life_event_person,0,family_relation_to_main_life_event_person
2,family_relation,main_life_event_person,0,family_relation_to_main_life_event_person
3,family_relation,family_relation,1,family_relation_to_family_relation
4,family_relation,family_relation,1,family_relation_to_family_relation


## Final Validation

In [26]:
similarity_feature_columns = [
    "first_name_similarity",
    "lastname_similarity",
    "full_name_similarity",

    "gender_similarity",

    "birth_year_difference",
    "death_year_difference",
    "event_year_difference",

    "event_place_similarity",
    "birth_place_similarity",
    "death_place_similarity",
    "resident_in_similarity",

    "same_role_group",
    "role_pair"
]

print("Similarity feature columns:")
for col in similarity_feature_columns:
    print("-", col)

Similarity feature columns:
- first_name_similarity
- lastname_similarity
- full_name_similarity
- gender_similarity
- birth_year_difference
- death_year_difference
- event_year_difference
- event_place_similarity
- birth_place_similarity
- death_place_similarity
- resident_in_similarity
- same_role_group
- role_pair


In [27]:
print("Final shape after adding similarity features:")
print(candidate_pairs.shape)

candidate_pairs[similarity_feature_columns].head()

Final shape after adding similarity features:
(1257724, 59)


,first_name_similarity,lastname_similarity,full_name_similarity,gender_similarity,birth_year_difference,death_year_difference,event_year_difference,event_place_similarity,birth_place_similarity,death_place_similarity,resident_in_similarity,same_role_group,role_pair
0,36.363636,100.0,72.000000,1,NaN,NaN,48.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
1,42.857143,100.0,71.428571,0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,0,family_relation_to_main_life_event_person
2,36.363636,100.0,72.000000,1,NaN,NaN,48.0,0.0,NaN,NaN,NaN,0,family_relation_to_main_life_event_person
3,46.153846,100.0,74.074074,1,NaN,NaN,47.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
4,46.153846,100.0,74.074074,1,NaN,NaN,106.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation


In [29]:
preview_columns = [
    "persona_idno_1",
    "persona_idno_2",
    "blocking_rules_matched",
    "blocking_rule_count",

    "name_clean_1",
    "name_clean_2",
    "lastname_clean_1",
    "lastname_clean_2",

    "first_name_similarity",
    "lastname_similarity",
    "full_name_similarity",

    "gender_1",
    "gender_2",
    "gender_similarity",

    "birth_year_1",
    "birth_year_2",
    "birth_year_difference",

    "death_year_1",
    "death_year_2",
    "death_year_difference",

    "event_year_1",
    "event_year_2",
    "event_year_difference",

    "event_place_similarity",
    "birth_place_similarity",
    "death_place_similarity",
    "resident_in_similarity",

    "same_role_group",
    "role_pair"
]

candidate_pairs[preview_columns].head(20)

,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count,name_clean_1,name_clean_2,lastname_clean_1,lastname_clean_2,first_name_similarity,lastname_similarity,...,death_year_difference,event_year_1,event_year_2,event_year_difference,event_place_similarity,birth_place_similarity,death_place_similarity,resident_in_similarity,same_role_group,role_pair
0,persona-10,persona-10078,rule1_lastname_initial,1,jacinto,jose,quispe,quispe,36.363636,100.0,...,NaN,1790.0,1838.0,48.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
1,persona-10,persona-1008,rule1_lastname_initial,1,jacinto,juliana,quispe,quispe,42.857143,100.0,...,NaN,1790.0,1793.0,3.0,NaN,NaN,NaN,NaN,0,family_relation_to_main_life_event_person
2,persona-10,persona-10081,rule1_lastname_initial,1,jacinto,jose,quispe,quispe,36.363636,100.0,...,NaN,1790.0,1838.0,48.0,0.0,NaN,NaN,NaN,0,family_relation_to_main_life_event_person
3,persona-10,persona-10320,rule1_lastname_initial,1,jacinto,julian,quispe,quispe,46.153846,100.0,...,NaN,1790.0,1837.0,47.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
4,persona-10,persona-10366,rule1_lastname_initial,1,jacinto,julian,quispe,quispe,46.153846,100.0,...,NaN,1790.0,1896.0,106.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
5,persona-10,persona-1040,rule1_lastname_initial|rule2_lastname_firstnam...,3,jacinto,jacinto,quispe,quispe,100.000000,100.0,...,NaN,1790.0,1793.0,3.0,NaN,NaN,NaN,NaN,1,family_relation_to_family_relation
6,persona-10,persona-10494,rule1_lastname_initial,1,jacinto,juana,quispe,quispe,50.000000,100.0,...,NaN,1790.0,1896.0,106.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
7,persona-10,persona-10507,rule1_lastname_initial,1,jacinto,juana,quispe,quispe,50.000000,100.0,...,NaN,1790.0,1896.0,106.0,0.0,NaN,NaN,NaN,0,family_relation_to_godparent
8,persona-10,persona-10601,rule1_lastname_initial,1,jacinto,jervacio,quispe,quispe,66.666667,100.0,...,NaN,1790.0,1896.0,106.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation
9,persona-10,persona-10883,rule1_lastname_initial,1,jacinto,jervacio,quispe,quispe,66.666667,100.0,...,NaN,1790.0,1899.0,109.0,0.0,NaN,NaN,NaN,1,family_relation_to_family_relation


In [32]:
candidate_pairs.to_parquet(output_path, index=False)